In [ ]:
# Import required libraries -- 00
import pennylane as qml
from pennylane import numpy as np
import plotly.express as plot

In [ ]:
USE_SIMULATOR = True

In [ ]:
# Create noise simulator -- 02
if USE_SIMULATOR:
    import qiskit_aer.noise as noise
    from qiskit_ibm_runtime.fake_provider import FakeAthensV2 as QCPU
    noise_model = noise.NoiseModel.from_backend(QCPU())


In [ ]:
# Assign simulator parameters -- 02
wires = [0, 1, 2, 3]
shots  = None or 500
if USE_SIMULATOR:
    dev = qml.device('qiskit.aer', wires=wires, noise_model=noise_model)
else:
    dev = qml.device('default.qubit', wires=wires)
np.set_printoptions(formatter={'all': lambda x: f"{x:.3f}" if np.iscomplexobj(x) or np.isrealobj(x) else str(x)})

In [ ]:
# Cooler print format -- 00
def probs_to_bits(probs, args:list=[]):
    from itertools import product
    for (arg, mode) in args:
        if arg not in probs or len(probs[arg]) == 0:
            continue
        if mode == 'bits':
            probs[arg] = {str(''.join(map(str, id))): np.float64(probs[arg][i]) for i, id in enumerate(product([0, 1], repeat=len(wires)))}
        if mode == 'int':
            probs[arg] = { i : np.complex64(probs[arg][i]) for i in range(0, len(probs[arg]))}
    return probs

In [ ]:
# Oracles declaration -- 03
def constant_oracle(seq = [1,2,3], res = 0, code = []):
    qml.X(res)

def balanced_oracle(seq = [1,2,3], res = 0, code = [1, 0, 1]):
    if code[0]:
        qml.X(seq[0])
    if code[1]:
        qml.X(seq[1])
    if code[2]:
        qml.X(seq[2])
    qml.CNOT([seq[0], res])
    qml.CNOT([seq[1], res])
    qml.CNOT([seq[2], res])
    if code[0]:
        qml.X(seq[0])
    if code[1]:
        qml.X(seq[1])
    if code[2]:
        qml.X(seq[2])


In [ ]:
# Circuit declaration and display -- 03
from types import FunctionType
@qml.qnode(dev, shots=shots)
def circuit(params = [], oracle : FunctionType = None, code = [1,0,1]):
    qml.X(0)
    qml.H(0)
    list(map(qml.H, [1,2,3]))
    oracle([1,2,3], 0, code)
    list(map(qml.H, [1,2,3]))
    qml.Z(0)
    return {
        'state': qml.state() if not USE_SIMULATOR else [],
        'probs': qml.probs([0, 1, 2, 3]),
        'shots': qml.sample() if shots else [],
        'test_res1': qml.probs([1]),
        'test_res2': qml.probs([1]),
        'test_res3': qml.probs([1])
    }
print(qml.draw(circuit)(oracle=constant_oracle))
print('\n')
print(qml.draw(circuit)(oracle=balanced_oracle, code = [1,1,1]))

In [ ]:
# Execute circuit -- 03
data1 = probs_to_bits(circuit(oracle=constant_oracle), [('state', 'int'), ('probs', 'bits')])
data2 = probs_to_bits(circuit(oracle=balanced_oracle), [('state', 'int'), ('probs', 'bits')])
{'data1': data1, 'data2': data2}

In [ ]:
# Result analisys and graph rendering -- 03
graph1 = plot.bar(x=data1['probs'].keys(), y=data1['probs'].values(), labels={"x": "Sequence", "y": "Occurrence"}, range_y=[0,1], height=500, width=800, text_auto=True)
graph1.show()
plot1 = plot.bar(x=[f'{i}' for i in range(1,4)], y=[data1[f'test_res{i}'][1] for i in range(1,4)], range_y=[0,1], height=500, width=800, text_auto=True, labels={"x": "qbit", "y": "Chance to read 1"})
plot1.show()

graph2 = plot.bar(x=data2['probs'].keys(), y=data2['probs'].values(), labels={"x": "Sequence", "y": "Occurrence"}, range_y=[0,1], height=500, width=800, text_auto=True)
graph2.show()
plot2 = plot.bar(x=[f'{i}' for i in range(1,4)], y=[data2[f'test_res{i}'][1] for i in range(1,4)], range_y=[0,1], height=500, width=800, text_auto=True, labels={"x": "qbit", "y": "Chance to read 1"})
plot2.show()

In [ ]:
# Test result: -- 03
for (n, data) in [('one', data1), ('two', data2)]:
    print(f"Oracle number {n} is", "balanced!" if data['test_res1'][0] < 0.5 else 'constant!', f"(test result for qbit 1: {data['test_res1'][0]})")